# Module 13: Advanced OOP Patterns

Split from `01_classes_and_oop.ipynb` for depth. Start with [Classes](./01_classes.ipynb) first.

**Navigation:** [Previous: Classes](./01_classes.ipynb) · [Topic overview](./01_classes_and_oop.ipynb) · [Next: Module 14](../14_Decorators_and_Context_Managers/01_decorators_and_context_managers.ipynb)

## Why advanced OOP patterns matter in bioinformatics

Once you can write a basic class, the next challenge is designing classes that compose well, scale to real datasets, and integrate cleanly with Python's built-in protocols. A `Seq` object that supports slicing (`seq[3:10]`), a `VariantSet` that can be indexed (`vs['rs123']`), or a `GeneDatabase` that acts like a dict — all require the patterns in this notebook.

## How to use this notebook

This notebook assumes you have worked through `01_classes.ipynb`. Run cells top to bottom. The design patterns here build incrementally — each section introduces a concept then shows a complete bioinformatics example.

## Advanced OOP topics covered

**1. `__getitem__`, `__setitem__`, `__contains__`** — make objects subscriptable
**2. `__call__`** — callable objects (function-like classes)
**3. `__slots__`** — memory-efficient objects for large collections
**4. Method Resolution Order (MRO)** — how Python resolves diamond inheritance
**5. Mixins** — composable behaviors without deep inheritance
**6. Duck typing and protocols** — Pythonic interfaces
**7. `__enter__` / `__exit__`** — context manager protocol on custom classes
**8. Complete design pattern: a FASTA database object**

---

## 1. `__getitem__`, `__setitem__`, `__contains__`: Subscriptable Objects

Implement `__getitem__` to allow `obj[key]` syntax, `__setitem__` for `obj[key] = val`, and `__contains__` for `key in obj`.

In [ ]:
class SequenceDatabase:
    """A dict-like container for biological sequences.
    
    Supports:
      - db['BRCA1']           -- retrieve by gene name
      - db['BRCA1'] = seq     -- store a sequence
      - 'BRCA1' in db         -- membership test
      - len(db)               -- number of sequences
      - for name in db:       -- iteration over names
    """

    def __init__(self):
        self._data = {}

    def __setitem__(self, name, sequence):
        self._data[name.upper()] = sequence.upper()

    def __getitem__(self, name):
        try:
            return self._data[name.upper()]
        except KeyError:
            raise KeyError(f"Gene '{name}' not in database. "
                           f"Available: {list(self._data.keys())[:5]}")

    def __contains__(self, name):
        return name.upper() in self._data

    def __len__(self):
        return len(self._data)

    def __iter__(self):
        return iter(self._data)

    def __repr__(self):
        return f"SequenceDatabase({len(self)} entries)"

    def gc_filter(self, min_gc=0.5):
        """Return a new SequenceDatabase with only high-GC sequences."""
        result = SequenceDatabase()
        for name, seq in self._data.items():
            gc = (seq.count('G') + seq.count('C')) / len(seq)
            if gc >= min_gc:
                result[name] = seq
        return result


# Build a sequence database
db = SequenceDatabase()
db['BRCA1'] = 'ATGCGATCGATCGCGATCGATCGATCGATCGATCGATCG'
db['TP53']  = 'GCGCGCGCGCGCATGATCGATCGATCGATCGATCG'
db['EGFR']  = 'ATATATATATATATATATATATCGATCGATCGATCGATCG'
db['MYC']   = 'GCGCGCGCGCGCGCGCGCGCGCGCGCATGATCGATCG'

print(db)
print(f"Has BRCA1: {'BRCA1' in db}")
print(f"Has KRAS:  {'KRAS' in db}")
print(f"BRCA1: {db['BRCA1'][:20]}...")

# Iterate
print("\nAll genes:")
for name in db:
    print(f"  {name}: {len(db[name])} bp")

# Filter
high_gc = db.gc_filter(min_gc=0.55)
print(f"\nHigh-GC sequences (>=55%): {list(high_gc)}")

---

## 2. Sliceable Objects: `__getitem__` with Slices

When `__getitem__` receives a `slice` object, you can make your sequence type support slicing just like strings and lists.

In [ ]:
class BioSeq:
    """A sliceable biological sequence.
    
    Supports:
      - seq[3]      -- single base by index
      - seq[3:10]   -- slice
      - seq[::3]    -- every third base (codon starts)
      - len(seq), str(seq), repr(seq)
    """

    def __init__(self, sequence, name="unnamed"):
        self.sequence = sequence.upper()
        self.name = name

    def __getitem__(self, key):
        result = self.sequence[key]
        if isinstance(key, slice):
            return BioSeq(result, name=f"{self.name}[slice]")
        return result  # single character

    def __len__(self):
        return len(self.sequence)

    def __str__(self):
        return self.sequence

    def __repr__(self):
        return f"BioSeq({self.name!r}, {len(self)} bp)"

    def __add__(self, other):
        """Concatenate two BioSeq objects."""
        new_name = f"{self.name}+{other.name}"
        return BioSeq(str(self) + str(other), name=new_name)

    def gc_content(self):
        return (self.sequence.count('G') + self.sequence.count('C')) / len(self)

    def codons(self):
        """Yield codons from reading frame 0."""
        for i in range(0, len(self) - 2, 3):
            yield self.sequence[i:i+3]


# Test BioSeq
seq = BioSeq("ATGGCCGATCGATCGTAGCGA", name="test_gene")
print(repr(seq))
print(f"First base:  {seq[0]}")
print(f"Last base:   {seq[-1]}")
print(f"First codon: {seq[0:3]}")
print(f"GC:          {seq.gc_content():.1%}")

# Slicing returns another BioSeq
fragment = seq[3:12]
print(f"\nFragment: {repr(fragment)}")
print(f"Fragment sequence: {fragment}")

# Concatenation
seq2 = BioSeq("GCTAGCTAGC", name="exon2")
joined = seq + seq2
print(f"\nJoined: {repr(joined)}")
print(f"Codons: {list(joined.codons())}")

---

## 3. `__call__`: Callable Objects

Adding `__call__` to a class makes its instances work like functions. This is useful for objects that need to maintain state between calls (like a scorer or a filter).

In [ ]:
class MotifScorer:
    """A callable that scores a sequence window for a specific motif.
    
    Usage:
        scorer = MotifScorer('TATAAA', mismatch_penalty=2)
        score = scorer('TATAAAGCGT')  # called like a function
    """

    def __init__(self, motif, mismatch_penalty=1):
        self.motif = motif.upper()
        self.mismatch_penalty = mismatch_penalty
        self._calls = 0  # track usage

    def __call__(self, window):
        """Score a window: 0 = perfect match, negative = mismatches."""
        self._calls += 1
        window = window.upper()[:len(self.motif)]
        if len(window) < len(self.motif):
            return -len(self.motif) * self.mismatch_penalty
        score = 0
        for a, b in zip(self.motif, window):
            if a != b:
                score -= self.mismatch_penalty
        return score

    def scan(self, sequence, threshold=0):
        """Find all positions where the score meets the threshold."""
        sequence = sequence.upper()
        hits = []
        for i in range(len(sequence) - len(self.motif) + 1):
            window = sequence[i:i + len(self.motif)]
            score = self(window)
            if score >= threshold:
                hits.append((i, window, score))
        return hits

    def __repr__(self):
        return f"MotifScorer(motif={self.motif!r}, calls={self._calls})"


# TATA box scanner
tata_scorer = MotifScorer('TATAAA', mismatch_penalty=2)
dna = "GCGATCGTATAATGCGGTATAAAGCGATCGATATAAGCG"

hits = tata_scorer.scan(dna, threshold=-2)  # allow 1 mismatch
print(f"TATA-box candidates in {dna}:")
for pos, window, score in hits:
    print(f"  pos {pos}: {window}  score={score}")

print(f"\n{repr(tata_scorer)}")

# Can be used as a key function or passed to map/filter
sequences = ['TATAAAGCG', 'GCATCGATCG', 'ATATAAAATG', 'CGCGATCG']
scored = [(seq, tata_scorer(seq)) for seq in sequences]
best = max(scored, key=lambda x: x[1])
print(f"\nBest TATA match: {best[0]} (score={best[1]})")

---

## 4. `__slots__`: Memory-Efficient Objects

By default, each Python object stores its attributes in a `__dict__`. For large collections (millions of SNPs, millions of k-mers), the per-object `__dict__` overhead is significant. Declaring `__slots__` eliminates the `__dict__` and reduces memory by ~40-60%.

In [ ]:
import sys

class VariantDict:
    """A SNP/variant record without __slots__ (normal class)."""
    def __init__(self, chrom, pos, ref, alt, qual):
        self.chrom = chrom
        self.pos = pos
        self.ref = ref
        self.alt = alt
        self.qual = qual


class VariantSlots:
    """A SNP/variant record with __slots__ (memory-optimized)."""
    __slots__ = ('chrom', 'pos', 'ref', 'alt', 'qual')

    def __init__(self, chrom, pos, ref, alt, qual):
        self.chrom = chrom
        self.pos = pos
        self.ref = ref
        self.alt = alt
        self.qual = qual


# Compare memory usage
n = 10_000
normal_variants = [VariantDict('chr17', i, 'A', 'G', 40.0) for i in range(n)]
slotted_variants = [VariantSlots('chr17', i, 'A', 'G', 40.0) for i in range(n)]

normal_size  = sum(sys.getsizeof(v) + sys.getsizeof(v.__dict__) for v in normal_variants)
slotted_size = sum(sys.getsizeof(v) for v in slotted_variants)

print(f"Normal (with __dict__): {normal_size / 1024:.1f} KB for {n:,} objects")
print(f"Slotted (__slots__):    {slotted_size / 1024:.1f} KB for {n:,} objects")
print(f"Memory reduction: {(1 - slotted_size/normal_size):.0%}")

# __slots__ objects work the same way
v = VariantSlots('chr17', 43045629, 'G', 'A', 99.5)
print(f"\nVariant: {v.chrom}:{v.pos} {v.ref}>{v.alt} QUAL={v.qual}")

# But you cannot add arbitrary attributes
try:
    v.annotation = "pathogenic"
except AttributeError as e:
    print(f"Cannot add new attributes: {e}")

---

## 5. Method Resolution Order (MRO) and Mixins

Python uses the **C3 linearization** algorithm to determine the order in which base classes are searched when a method is called. This matters for multiple inheritance.

**Mixins** are small, single-purpose classes intended to be mixed into other classes. They add a specific capability without defining a complete object on their own.

In [ ]:
from abc import ABC, abstractmethod


class BioSequenceMixin:
    """Mixin providing common sequence utilities.
    Requires self.sequence to be defined by the host class.
    """

    def gc_content(self):
        seq = self.sequence.upper()
        return (seq.count('G') + seq.count('C')) / len(seq)

    def nucleotide_counts(self):
        seq = self.sequence.upper()
        return {nt: seq.count(nt) for nt in 'ATGC'}

    def is_valid(self, valid_chars):
        return set(self.sequence.upper()) <= set(valid_chars.upper())


class FASTASerializableMixin:
    """Mixin that adds FASTA export capability.
    Requires self.name and self.sequence.
    """

    def to_fasta(self, line_width=60):
        lines = [f'>{self.name}']
        seq = self.sequence
        for i in range(0, len(seq), line_width):
            lines.append(seq[i:i + line_width])
        return '\n'.join(lines)


class ReversibleMixin:
    """Mixin for sequences that have a reverse complement."""
    _COMPLEMENT = str.maketrans('ATGCatgc', 'TACGtacg')

    def reverse_complement(self):
        return self.sequence.translate(self._COMPLEMENT)[::-1]


class DNARecord(BioSequenceMixin, FASTASerializableMixin, ReversibleMixin):
    """A concrete DNA sequence combining multiple mixins."""

    def __init__(self, name, sequence):
        self.name = name
        self.sequence = sequence.upper()

    def __repr__(self):
        return f"DNARecord({self.name!r}, {len(self.sequence)} bp)"


# MRO: order in which Python searches for methods
print("MRO:", [cls.__name__ for cls in DNARecord.__mro__])

# Use the class
gene = DNARecord("BRCA1_exon11", "ATGCGATCGATCGCGATCGATCGATCGATCGATCGATCGATCG")
print(f"\n{repr(gene)}")
print(f"GC content: {gene.gc_content():.1%}")
print(f"Valid DNA:  {gene.is_valid('ATGCN')}")
print(f"Rev comp:   {gene.reverse_complement()[:20]}...")
print(f"\nFASTA format:\n{gene.to_fasta(line_width=20)}")

---

## 6. Duck Typing and Protocols

Python does not require formal interface declarations. Any object that provides the right methods is accepted — this is **duck typing**: "if it walks like a duck and quacks like a duck, it's a duck."

For more formal type checking without ABCs, Python 3.8+ provides `typing.Protocol`.

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Alignable(Protocol):
    """Protocol for objects that can be pairwise aligned."""
    
    @property
    def sequence(self) -> str: ...
    
    @property
    def name(self) -> str: ...


def pairwise_identity(seq1, seq2):
    """Compute pairwise identity between two Alignable objects.
    Works with any object that has .sequence and .name — no inheritance required.
    """
    s1 = seq1.sequence.upper()
    s2 = seq2.sequence.upper()
    min_len = min(len(s1), len(s2))
    matches = sum(a == b for a, b in zip(s1, s2))
    return matches / min_len if min_len > 0 else 0.0


# Works with DNARecord from above
g1 = DNARecord("Human_BRCA1", "ATGCGATCGATCGATCGATCG")
g2 = DNARecord("Mouse_BRCA1", "ATGCGATCGATCGATCGATTG")

# Also works with a completely unrelated class
class RNAFrag:
    """A minimal RNA fragment — not inherited from anything special."""
    def __init__(self, name, sequence):
        self.name = name
        self.sequence = sequence.upper().replace('T', 'U')

rna = RNAFrag("mRNA_1", "AUGCGAUCGAUCGAUCGAUUG")

# Runtime protocol check
print(f"DNARecord is Alignable: {isinstance(g1, Alignable)}")
print(f"RNAFrag is Alignable:   {isinstance(rna, Alignable)}")

# Both work with pairwise_identity
identity = pairwise_identity(g1, g2)
print(f"\nHuman vs Mouse BRCA1 identity: {identity:.1%}")

# Process any collection of Alignable objects
sequences = [g1, g2, DNARecord("Chimp_BRCA1", "ATGCGATCGATCGATCGATCG")]
for i, s1 in enumerate(sequences):
    for s2 in sequences[i+1:]:
        id_val = pairwise_identity(s1, s2)
        print(f"  {s1.name} vs {s2.name}: {id_val:.1%}")

---

## 7. Context Manager Protocol on Custom Classes

Any class with `__enter__` and `__exit__` methods can be used with `with`. This is how database connections, file handles, and resource managers work.

In [ ]:
import sqlite3
from contextlib import contextmanager


class GenomicDatabase:
    """An SQLite-backed genomic database with context manager support."""

    def __init__(self, db_path=':memory:'):
        self.db_path = db_path
        self.conn = None

    def __enter__(self):
        self.conn = sqlite3.connect(self.db_path)
        self.conn.row_factory = sqlite3.Row  # named columns
        self._setup_schema()
        return self

    def __exit__(self, exc_type, exc_val, tb):
        if exc_type is None:
            self.conn.commit()
        else:
            self.conn.rollback()
        self.conn.close()
        self.conn = None
        return False  # re-raise exceptions

    def _setup_schema(self):
        self.conn.execute('''
            CREATE TABLE IF NOT EXISTS genes (
                gene_id   TEXT PRIMARY KEY,
                symbol    TEXT NOT NULL,
                chrom     TEXT,
                start_pos INTEGER,
                end_pos   INTEGER,
                strand    TEXT
            )
        ''')

    def insert_gene(self, gene_id, symbol, chrom, start, end, strand):
        self.conn.execute(
            'INSERT OR REPLACE INTO genes VALUES (?,?,?,?,?,?)',
            (gene_id, symbol, chrom, start, end, strand)
        )

    def query(self, sql, params=()):
        return list(self.conn.execute(sql, params))

    def genes_on_chrom(self, chrom):
        rows = self.query('SELECT symbol, start_pos, end_pos FROM genes WHERE chrom=?', (chrom,))
        return rows


# Use with context manager
with GenomicDatabase(':memory:') as db:
    # Insert some well-known cancer genes
    db.insert_gene('ENSG00000141510', 'TP53',  'chr17', 7_661_779, 7_687_550, '-')
    db.insert_gene('ENSG00000012048', 'BRCA1', 'chr17', 43_044_295, 43_170_245, '-')
    db.insert_gene('ENSG00000146648', 'EGFR',  'chr7',  55_086_714, 55_324_313, '+')
    db.insert_gene('ENSG00000136997', 'MYC',   'chr8',  127_735_434, 127_742_951, '+')

    genes_chr17 = db.genes_on_chrom('chr17')
    print("Genes on chr17:")
    for row in genes_chr17:
        length = row['end_pos'] - row['start_pos']
        print(f"  {row['symbol']}: {row['start_pos']:,}–{row['end_pos']:,} ({length:,} bp)")

    total = db.query('SELECT COUNT(*) AS n FROM genes')[0]['n']
    print(f"\nTotal genes in database: {total}")

print("\nConnection automatically closed and committed.")

---

## 8. Complete Design Pattern: A FASTA Database Object

Combining everything: subscript access, iteration, context manager, mixins, and duck typing.

In [ ]:
from dataclasses import dataclass, field
from typing import Iterator, Optional
import re


@dataclass
class FASTARecord:
    """A single FASTA record."""
    seq_id: str
    description: str
    sequence: str

    @property
    def name(self):
        return self.seq_id

    def gc_content(self):
        s = self.sequence.upper()
        return (s.count('G') + s.count('C')) / len(s) if s else 0.0

    def length(self):
        return len(self.sequence)

    def to_fasta(self, line_width=60):
        lines = [f'>{self.seq_id} {self.description}']
        for i in range(0, len(self.sequence), line_width):
            lines.append(self.sequence[i:i+line_width])
        return '\n'.join(lines)


class FASTADatabase:
    """An in-memory FASTA database with dict-like access and iteration.
    
    Supports:
      db['BRCA1']        -- retrieve by ID or gene name
      'BRCA1' in db      -- membership test
      for record in db:  -- iterate over all records
      len(db)            -- number of sequences
      db.filter(gc>0.5)  -- returns a new FASTADatabase
    """

    def __init__(self):
        self._records: dict[str, FASTARecord] = {}

    def add(self, record: FASTARecord):
        self._records[record.seq_id] = record

    def load_fasta_text(self, text: str):
        """Parse a FASTA-format string and add all records."""
        current_id = None
        current_desc = ''
        seq_parts = []
        for line in text.splitlines():
            line = line.strip()
            if line.startswith('>'):
                if current_id:
                    self.add(FASTARecord(current_id, current_desc, ''.join(seq_parts)))
                parts = line[1:].split(None, 1)
                current_id = parts[0]
                current_desc = parts[1] if len(parts) > 1 else ''
                seq_parts = []
            elif line:
                seq_parts.append(line)
        if current_id:
            self.add(FASTARecord(current_id, current_desc, ''.join(seq_parts)))

    def __getitem__(self, key: str) -> FASTARecord:
        if key in self._records:
            return self._records[key]
        raise KeyError(f"Sequence '{key}' not found")

    def __contains__(self, key: str) -> bool:
        return key in self._records

    def __len__(self) -> int:
        return len(self._records)

    def __iter__(self) -> Iterator[FASTARecord]:
        return iter(self._records.values())

    def __repr__(self):
        return f"FASTADatabase({len(self)} sequences)"

    def filter(self, predicate) -> 'FASTADatabase':
        """Return a new FASTADatabase containing only records satisfying predicate."""
        result = FASTADatabase()
        for record in self:
            if predicate(record):
                result.add(record)
        return result

    def summary(self):
        total = len(self)
        if total == 0:
            print("Empty database")
            return
        lengths = [r.length() for r in self]
        gcs = [r.gc_content() for r in self]
        print(f"Records: {total}")
        print(f"Length:  min={min(lengths)}, max={max(lengths)}, "
              f"mean={sum(lengths)/len(lengths):.0f}")
        print(f"GC%%:    min={min(gcs):.1%%}, max={max(gcs):.1%%}, "
              f"mean={sum(gcs)/len(gcs):.1%%}")


# Build a database from inline FASTA
fasta_text = """>BRCA1 Breast cancer type 1 susceptibility protein
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA
ATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGAC
>TP53 Cellular tumor antigen p53
ATGGAGGAGCCGCAGTCAGATCCTAGCAGAGCATCTCCGAGGCCTGGCTCCCCTGTGCCC
TGTGGGCTTCGAGATCCAGAGGCAGAGCCAGCTTTCCATGGCTGCCAAGCTGTGGCCCTG
>EGFR Epidermal growth factor receptor
ATGCGACCCTCCGGGACGGCCGGGGCAGCGCTCCTGGCGCTGCTGGCTGCGCTCTGCCCG
GCGAGTCGGGCTCTGGAGGAAAAGAAAGTTCGAATCGTTGTTGATAGCGATGATGAGCAG
"""

db = FASTADatabase()
db.load_fasta_text(fasta_text)

print(repr(db))
print()
db.summary()

# Access by ID
brca1 = db['BRCA1']
print(f"\nBRCA1: {brca1.length()} bp, GC={brca1.gc_content():.1%}")
print(f"BRCA1 in db: {'BRCA1' in db}")
print(f"KRAS  in db: {'KRAS' in db}")

# Filter
long_seqs = db.filter(lambda r: r.length() > 180)
print(f"\nSequences >180 bp: {list(r.seq_id for r in long_seqs)}")

# Iterate and format as FASTA
print("\nAll sequences (first 60 bp each):")
for record in db:
    print(f"  {record.seq_id}: {record.sequence[:60]}...")

---

## Exercises

### Exercise 1: Extend FASTADatabase with search

Add a `search(pattern)` method to `FASTADatabase` that accepts a regex pattern and returns a new `FASTADatabase` containing records whose sequence matches the pattern anywhere.

In [ ]:
# Exercise 1: Your solution
import re

class FASTADatabaseV2(FASTADatabase):
    def search(self, pattern):
        """Return sequences matching the regex pattern."""
        # Your code here
        pass

# Test:
# db2 = FASTADatabaseV2()
# db2.load_fasta_text(fasta_text)
# atg_seqs = db2.search(r'ATG[ATGC]{6}ATG')
# print(f"Sequences with double ATG motif: {[r.seq_id for r in atg_seqs]}")

In [ ]:
# --- Solution ---
import re

class FASTADatabaseV2(FASTADatabase):
    def search(self, pattern):
        """Return sequences matching the regex pattern."""
        compiled = re.compile(pattern, re.IGNORECASE)
        return self.filter(lambda r: bool(compiled.search(r.sequence)))


db2 = FASTADatabaseV2()
db2.load_fasta_text(fasta_text)
atg_seqs = db2.search(r'ATG[ATGC]{6}ATG')
print(f"Sequences with double ATG motif: {[r.seq_id for r in atg_seqs]}")

### Exercise 2: `__slots__` SNP record

Create a `SNPRecord` class using `__slots__` with fields: `rsid`, `chrom`, `pos`, `ref`, `alt`, `af` (allele frequency). Verify you can create 100,000 objects and compare memory with a normal class.

In [ ]:
# Exercise 2: Your solution

class SNPRecord:
    # Your code here
    pass

# Test:
# snp = SNPRecord('rs123456', 'chr1', 1000000, 'A', 'G', 0.15)
# print(snp.rsid, snp.pos, snp.af)

In [ ]:
# --- Solution ---
import sys

class SNPRecord:
    __slots__ = ('rsid', 'chrom', 'pos', 'ref', 'alt', 'af')

    def __init__(self, rsid, chrom, pos, ref, alt, af):
        self.rsid = rsid
        self.chrom = chrom
        self.pos = pos
        self.ref = ref
        self.alt = alt
        self.af = af


class SNPRecordNormal:
    def __init__(self, rsid, chrom, pos, ref, alt, af):
        self.rsid = rsid
        self.chrom = chrom
        self.pos = pos
        self.ref = ref
        self.alt = alt
        self.af = af


n = 100_000
slotted = [SNPRecord(f'rs{i}', 'chr1', i, 'A', 'G', 0.1) for i in range(n)]
normal  = [SNPRecordNormal(f'rs{i}', 'chr1', i, 'A', 'G', 0.1) for i in range(n)]

sz_slotted = sum(sys.getsizeof(x) for x in slotted)
sz_normal  = sum(sys.getsizeof(x) + sys.getsizeof(x.__dict__) for x in normal)

print(f"Slotted: {sz_slotted/1024:.0f} KB")
print(f"Normal:  {sz_normal/1024:.0f} KB")
print(f"Savings: {(1 - sz_slotted/sz_normal):.0%}")

### Exercise 3: Callable Restriction Enzyme

Create a `RestrictionEnzyme` class with `__call__` that takes a DNA sequence and returns a list of cut positions. The cut position for EcoRI (`GAATTC`) is after position 1 (G^AATTC).

In [ ]:
# Exercise 3: Your solution
class RestrictionEnzyme:
    pass

# Test:
# ecori = RestrictionEnzyme('EcoRI', 'GAATTC', cut=1)
# seq = "ATGCGAATTCGATCGAATTCATCG"
# cuts = ecori(seq)
# print(f"EcoRI cuts at: {cuts}")
# fragments = ecori.digest(seq)
# print(f"Fragments: {fragments}")

In [ ]:
# --- Solution ---
class RestrictionEnzyme:
    def __init__(self, name, recognition_site, cut):
        """
        name: enzyme name (e.g. 'EcoRI')
        recognition_site: recognition sequence (e.g. 'GAATTC')
        cut: cut position within site (0-indexed, e.g. 1 for G^AATTC)
        """
        self.name = name
        self.recognition_site = recognition_site.upper()
        self.cut = cut

    def __call__(self, sequence):
        """Return list of cut positions (in genomic coordinates)."""
        sequence = sequence.upper()
        site = self.recognition_site
        cut_positions = []
        start = 0
        while True:
            pos = sequence.find(site, start)
            if pos == -1:
                break
            cut_positions.append(pos + self.cut)
            start = pos + 1
        return cut_positions

    def digest(self, sequence):
        """Return list of fragments after digestion."""
        sequence = sequence.upper()
        cuts = [0] + self(sequence) + [len(sequence)]
        return [sequence[cuts[i]:cuts[i+1]] for i in range(len(cuts)-1)]

    def __repr__(self):
        return f"RestrictionEnzyme({self.name!r}, site={self.recognition_site!r})"


ecori  = RestrictionEnzyme('EcoRI',   'GAATTC', cut=1)
bamhi  = RestrictionEnzyme('BamHI',   'GGATCC', cut=1)
hindiii = RestrictionEnzyme('HindIII', 'AAGCTT', cut=1)

seq = "ATGCGAATTCGATCGAATTCATCGGGATCCATCGAAGCTTATCG"
print(f"Sequence: {seq}")
print(f"EcoRI  cuts at: {ecori(seq)}")
print(f"BamHI  cuts at: {bamhi(seq)}")
print(f"HindIII cuts at: {hindiii(seq)}")

print(f"\nEcoRI fragments: {ecori.digest(seq)}")

---

## Summary

```
ADVANCED OOP PATTERNS
+------------------------------------------------------------------+
|  __getitem__/__setitem__/__contains__  Subscriptable objects     |
|  __call__                              Callable objects (state)  |
|  __slots__                             Memory-efficient records   |
|  MRO / mixins                          Composable behaviors      |
|  Protocol / duck typing                Interface without ABCs    |
|  __enter__ / __exit__                  Resource management       |
+------------------------------------------------------------------+

Bioinformatics patterns:
  SequenceDatabase[name]    -- dict-like access to sequences
  BioSeq[3:10]              -- slice DNA/protein sequences
  MotifScorer('TATA')(seq)  -- stateful callable scorer
  SNPRecord with __slots__  -- memory-efficient for millions of SNPs
  FASTADatabase.filter(fn)  -- functional filtering
```

---

[← Previous: Classes](./01_classes.ipynb) | [Next: Module 14: Decorators and Context Managers →](../14_Decorators_and_Context_Managers/01_decorators_and_context_managers.ipynb)